# Deep Clustering (SimCLR) для СЭМ-изображений

### Шаг 1: Подключите GPU
`Среда выполнения` -> `Сменить среду выполнения` -> T4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp "/content/drive/MyDrive/diploma_colab.zip" /content/
!unzip -qo /content/diploma_colab.zip -d /content/diploma_project/
!pip install -q umap-learn

---
## Обучение SimCLR
Выберите ОДНУ из ячеек ниже:
- **3a** — первый запуск с нуля
- **3b** — продолжение после обрыва (стандартные параметры)
- **3c** — продолжение с агрессивными параметрами (batch=128, temp=0.2)

In [ ]:
# === 3a: ПЕРВЫЙ ЗАПУСК (с нуля, стандартные параметры) ===
!python /content/diploma_project/src/models/deep_clustering/train.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints/ \
    --epochs 50 \
    --batch_size 64 \
    --workers 2

In [ ]:
# === 3b: ПРОДОЛЖЕНИЕ (стандартные параметры) ===
LAST_COMPLETED_EPOCH = 19  # <-- Поменяй на номер последнего чекпоинта!
REMAINING_EPOCHS = 50 - LAST_COMPLETED_EPOCH

!python /content/diploma_project/src/models/deep_clustering/train.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints/ \
    --epochs {REMAINING_EPOCHS} \
    --batch_size 64 \
    --workers 2 \
    --resume /content/drive/MyDrive/diploma_checkpoints/simclr_resnet50_epoch_{LAST_COMPLETED_EPOCH}.pth \
    --start_epoch {LAST_COMPLETED_EPOCH}

In [ ]:
# === 3c: АГРЕССИВНЫЕ ПАРАМЕТРЫ (batch=128, temp=0.2) ===
# Запускаем НОВЫЙ эксперимент — чекпоинты сохраняются в отдельную папку!
!python /content/diploma_project/src/models/deep_clustering/train.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints_v2/ \
    --epochs 30 \
    --batch_size 128 \
    --temperature 0.2 \
    --workers 2

---
## Обучение BYOL (альтернативная архитектура)
BYOL не использует негативные пары и не зависит от размера батча.

In [ ]:
# === 4: BYOL обучение ===
!python /content/diploma_project/src/models/deep_clustering/train_byol.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints_byol/ \
    --epochs 30 \
    --batch_size 64 \
    --workers 2

---
## Извлечение эмбеддингов и сравнение

In [ ]:
# Укажи путь к лучшему чекпоинту (SimCLR v1, v2, или BYOL)
CHECKPOINT = '/content/drive/MyDrive/diploma_checkpoints/simclr_resnet50_epoch_50.pth'
# CHECKPOINT = '/content/drive/MyDrive/diploma_checkpoints_v2/simclr_resnet50_epoch_30.pth'
# CHECKPOINT = '/content/drive/MyDrive/diploma_checkpoints_byol/byol_resnet50_epoch_30.pth'

!python /content/diploma_project/src/models/deep_clustering/extract_simclr_embeddings.py \
    --checkpoint {CHECKPOINT} \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_results/ \
    --batch_size 128 \
    --workers 2

In [ ]:
from IPython.display import Image, display
display(Image('/content/drive/MyDrive/diploma_results/baseline_vs_simclr_umap.png'))